# Una mica de resum

---

La metrica promitjada $\bar{S}$ es defineix com:

$$
\bar{S} = \frac{1}{N} \sum_{i} S_i
$$

on cada $S_i$ es la métrica pèr cada replicat i $N$ es el número de replicats. Calculem $S_i$ de la següent manera:

$$
S_i = \text{PDI}_i \times S_{i}^\text{opt}
$$

Es a dir, el producte del PDI i un valor que ve de la convolució optima, $S_{i}^\text{opt}$, pel replicat $i$-éssim. Formalment, es defineix $S_{i}^\text{opt}$ com:

$$
S_{i}^\text{opt} = \min_{\mu_i,\sigma_i,A_i} \Bigg[\int_{-\infty}^{+\infty} |f_i(x) - g(x; \mu_i,\sigma_i, A_i)|^2 dx \Bigg]
$$

on $f_i(x)$ és la distribució de tamanys mesurada pel DLS i $g(x; \mu_i,\sigma_i, A_i)$ es defineix

$$
g(x; \mu_i,\sigma_i, A_i) = A_i \times e^{-\frac{(x-\mu_i)^2}{2 \sigma_i^2}}
$$

Traduït al català: la integral, $I=\int |\dots|^2 dx$, mesura la distància al quadrat entre la mesura $f_i(x)$ del DLS i $g(·)$. L'operador $\min_{\mu_i,\sigma_i,A_i}[·]$ busca el valor mínim de $I$ variant els parámetres de la gaussiana $\mu_i$, $\sigma_i$ i $A_i$. Per tant, $S_{i}^\text{opt}$ és la distáncia mímina entre la mesura del DLS i la millor $g(·)$.


# QUE FEM AQUÍ:
---

* Buscar el millor conjunt $(\mu_i,\sigma_i, A_i)$ es pot fer per minims quadrats amb `scipy.optimize.least_squares`.

* En el códi d'aquest notebook es constrieuxen tres "mesures" *ad hoc* i es defineix al funció `multi_least_squares` per calcular cada $S_i$.

* es fan dos exemples: primerament sense acotar, es a dir, amb $\mu_i, \sigma_i \in (-\infty, \infty)$. Per últim acotant $\mu_i \in (\mu_i^\text{min}, \mu_i^\text{max})$ i $\sigma_i \in (\sigma_i^\text{min}, \sigma_i^\text{max})$

* No es calcula $\bar{S}$, perque no es disposa del PDI.



In [4]:
pip install --upgrade pip 

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 3.5 MB/s eta 0:00:01
   ----------------------- ---------------- 1.0/1.8 MB 2.9 MB/s eta 0:00:01
   ----------------------------- ---------- 1.3/1.8 MB 2.4 MB/s eta 0:00:01
   ----------------------------------- ---- 1.6/1.8 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 2.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.1
    Uninstalling pip-25.1:
      Successfully uninstalled pip-25.1
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install numpy matplotlib scipy

Note: you may need to restart the kernel to use updated packages.


# Imports i funcions per calcular $g(·)$ i fer gràfics

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.integrate import trapezoid

In [7]:
def gaussian(x, mu, sigma, A=1):
    return A*np.exp(-(x - mu)**2 / (2 * sigma**2))

def plot_(x, lines, fit=None):
    fig, ax = plt.subplots(1, 1, figsize=(6,4))
    colors  = "blue red darkgreen orange".split()

    for c, line in zip(colors, lines):
        plt.plot(x, line, c=c, marker="." if len(line) < 100 else None)


    if fit is not None:
        for c, fit_i in zip(colors, fit):
            if fit_i is None: continue

            plt.plot(x, fit_i, ls="--", c="black", alpha=0.7)

    ax.grid(ls="--", lw=0.5, c="gray", alpha=0.5)

    plt.tight_layout()
    return fig, ax

In [9]:
def multi_least_squares(x, fs, mu_bounds, sigma_bounds=[1e-6, np.inf], mu_init=None, sigma_init=1, debug: int=0):

    def residuals_fn(params, x, f):
        mu, sigma = params

        if sigma <= 0:
            return np.inf * np.ones_like(x)

        g = gaussian(x, mu, sigma)
        A = np.dot(f, g) / np.dot(g, g)

        return f - A * g

    def get_in_bonds(x, x_bonds):
        if x is not None:
            return x

        finite_bonds = np.isfinite(x_bonds)

        if np.all(finite_bonds):
            return np.mean(x_bonds)

        if np.all(~finite_bonds):
            return 1

        return x_bonds[np.argwhere([True,False,True]).flatten()[0]]

    results = [
        least_squares(
            residuals_fn,
            x0     = [
                get_in_bonds(mu_init, mu_bounds),
                get_in_bonds(sigma_init, sigma_bounds),
            ],
            args   = [x, f],
            bounds = [
                [mu_bounds[0], sigma_bounds[0]],
                [mu_bounds[1], sigma_bounds[1]]
            ],
        )
        for f in fs
    ]

    gs = []
    for i, (res, f) in enumerate(zip(results, fs)):
        mu_opt, sigma_opt = res.x

        g_opt = gaussian(x, mu_opt, sigma_opt)
        A_opt = np.dot(f, g_opt) / np.dot(g_opt, g_opt)

        gs.append(gaussian(x, mu_opt, sigma_opt, A_opt))

        if debug > 0:
            print(f"mu    (f{i+1}) = {mu_opt   :.5f}")
            print(f"sigma (f{i+1}) = {sigma_opt:.5f}")
            print(f"A     (f{i+1}) = {A_opt    :.5f}")
            print()

    metrics = trapezoid((fs - gs)**2, x)

    if debug > 0:
        for i, m in enumerate(metrics):
            print(f"distance f({i+1}) = {m:.5e}")

    return np.array(gs), metrics

## Classes per Organitzar les Dades

He creat un sistema de classes per organitzar i emmagatzemar els resultats, per si et resulta util.

- Collection: Grup d'Experiments, per exemple, Sampling Melina. 
- Experiment: Resultats d'una mostra amb una composició definida, i un títol (ERM-xxx)
- Replica:    Una de les mesures associades a un experiment

El parseig de dades esta pensat per fitxers .csv amb una estructura identica a la del 
"MELINA_results_rawdata" que m'has enviat. Si cal modificar-ho, no hi ha problema, pero s'hauran de canviar coses

In [19]:
import numpy as np

################
## Collection ##
################
class Collection(object):
    def __init__(self, name: str):
        self.name        = name
        self.experiments = []
    
    def add_experiment(self, name, overwrite: bool=False, debug: int=0):
        found, experiment = self.find_experiment(name)
        if not found: 
            new_experiment = Experiment(name, collection=self)
            self.experiments.append(new_experiment)
            if debug > 0: print(f"Added Experiment {new_experiment.name}")
        elif found and overwrite:
                self.remove_experiment(name)
                new_experiment = Experiment(name, collection=self)
                self.experiments.append(new_experiment)
        else:
            if debug > 0: print(f"Experiment exists with same name={name}")
            if debug > 0: print(f"Use overwrite=True if you wish")
        return self.experiments

    def find_experiment(self, name, debug: int=0):
        if debug > 0: print(f"Searching experiment with name={name}")
        if "." in name: 
            number         = name.split(".")[1]
        elif "-" in name: 
            number         = name.split("-")[1]
        elif isinstance(name, int): 
            number = name 
        else:
            number = int(0)
        for exp in self.experiments:
            if debug > 0: print(f"Comparing {exp.name=} with {name=}")
            if debug > 0: print(f"And {exp.number=} with {number=}")
            if   exp.name    == name:   
                if debug > 0: print(f"Found same {exp.name=}")
                return True, exp
            elif exp.number  == number: 
                if debug > 0: print(f"Found same {exp.number=}")
                return True, exp
        return False, None

    def remove_experiment(self, name):
        name           = name
        for idx, exp in enumerate(self.experiments):
            if   exp.name    == name:   found = True; found_idx = idx
        if found:
            del self.experiments[found_idx]
            print(f"Experiment {name} removed from Collection {self.name}")

    def __repr__(self):
        to_print  = f"-- Melina DLS Collection --\n"
        to_print += f"NAME     : {self.name}\n"
        to_print += f"Experiments : {len(self.experiments)}\n"
        return to_print
    
################
## Experiment ##
################
class Experiment(object):
    def __init__(self, name, collection):
        self.name       = name
        self.code       = name.split("-")[0]
        self.number     = name.split("-")[1]
        self.replicas   = []
        self.collection = collection

    def add_replica(self, name, x_data, y_data, overwrite: bool=False, debug: int=0):
        if debug > 0: print(f"Adding replica with name: {name}")
        found, new_replica = self.find_replica(name, debug=debug)
        if not found: 
            if debug > 0: print(f"Creating and adding {name}")
            new_replica = Replica(name, experiment=self)
            new_replica.add_data(x_data, y_data, debug=debug) 
            self.replicas.append(new_replica)
        elif found and overwrite:
            if debug > 0: print(f"Removing, creating and adding {name}")
            self.remove_replica(name)
            new_replica = Replica(name, experiment=self)
            new_replica.add_data(x_data, y_data, debug=debug) 
            self.replicas.append(new_replica)
        else:
            if debug > 0: print(f"Replica {name} not added, returning existing one")
        return new_replica

    def find_replica(self, name, debug: int=0):
        if debug > 0: print(f"Searching Replica with name={name}")
        name           = name
        for rep in self.replicas:
            if   rep.name == name:   return True, rep 
        return False, None

    def remove_replica(self, name):
        name           = name
        code           = name.split(".")[0]
        number         = name.split(".")[1]
        for idx, rep in enumerate(self.replicas):
            if   rep.name    == name:   found = True; found_idx = idx
            elif rep.code    == code:   found = True; found_idx = idx 
            elif rep.number  == number: found = True; found_idx = idx
        if found:
            del self.replicas[found_idx]
            print(f"Replica {name} removed from Experiment {self.name}")

    @property
    def distance(self):
        if len(self.replicas) == 0: return None
        for rep in self.replicas:
            if not hasattr(rep,"distance"): return None
        return np.round(np.average([float(rep.distance) for rep in self.replicas]),4)
    
    def __repr__(self):
        to_print  = f"-- Melina DLS Experiment --\n"
        to_print += f"NAME       : {self.name}\n"
        to_print += f"CODE       : {self.code}\n"
        to_print += f"NUMBER     : {self.number}\n"
        to_print += f"REPLICAS   : {len(self.replicas)}\n"
        to_print += f"COLLECTION : {self.collection.name}\n"
        if self.distance is not None: to_print += f"DISTANCE : {self.distance}\n" 
        return to_print

#############
## Replica ##
#############
class Replica(object):
    def __init__(self, name, experiment):
        self.name           = name
        self.experiment     = experiment

        if   "." in name: 
            self.code           = name.split(".")[0]
            self.number         = name.split(".")[1]
        elif "-" in name: 
            self.code           = name.split("-")[0]
            self.number         = name.split("-")[1]
        else:
            self.code           = None
            self.number         = None

    def add_data(self, x_data, y_data, debug: int=0):
        import numpy as np
        self.x_data, self.y_data = curate_data(x_data[1:], y_data[1:], debug=debug)
        self.log_x_data = np.log10(self.x_data)

    def fit_data(self, mu_bounds=(0.1, 6), sigma_bounds=[1e-6, np.inf], mu_init=None, sigma_init=1, debug: int=0):
        assert hasattr(self,"log_x_data"), f"Missing X data. Import it"
        assert hasattr(self,"y_data"),     f"Missing Y data. Import it"
        self.mu_bounds       = mu_bounds
        self.sigma_bounds    = sigma_bounds
        self.mu_init         = mu_init
        self.sigma_init      = sigma_init
        self.fitted_data, metrics = multi_least_squares(self.log_x_data, np.array([self.y_data]), mu_bounds=mu_bounds, sigma_bounds=sigma_bounds, sigma_init=sigma_init, debug=debug) #, mu_bounds=x_range)
        self.distance = np.round(metrics[0],4)
        return self.distance

    def plot_fit(self, debug: int=0):
        if not hasattr(self,"fitted_data"): self.fit_data(debug=debug)
        fig, ax = plot_(self.log_x_data, np.array([self.y_data]), self.fitted_data)
        ax.axvline(self.mu_bounds[0], c="purple")
        ax.axvline(self.mu_bounds[-1], c="purple")
        plt.show()

    def __repr__(self):
        to_print  = f"-- Melina DLS Replica --\n"
        to_print += f"NAME       : {self.name}\n"
        to_print += f"CODE       : {self.code}\n"
        to_print += f"NUMBER     : {self.number}\n"
        to_print += f"DATA_LEN   : {len(self.x_data)}\n"
        if hasattr(self,"distance"): to_print += f"DISTANCE   : {self.distance}\n" 
        to_print += f"EXPERIMENT : {self.experiment.name}\n"
        to_print += f"COLLECTION : {self.experiment.collection.name}\n"
        return to_print



In [11]:
#####################
## Altres Funcions ##
#####################
def curate_data(x_data, y_data, debug: int=0):
    if debug > 0: print(f"CURATE_DATA. original lengths: X={len(x_data)}, Y={len(y_data)}")
    x_cur = []
    y_cur = []
    for x, y in zip(x_data, y_data):
        if x != '' and y != '' and float(x) not in x_cur:
            x_cur.append(float(x))
            y_cur.append(float(y))
    if debug > 0: print(f"CURATE_DATA. final lengths: X={len(x_data)}, Y={len(y_data)}")
    return x_cur, y_cur

##############
## Binaries ##
##############
def load_binary(pathfile):
    import pickle
    with open(pathfile, "rb") as pickle_file:
        binary = pickle.load(pickle_file)
    return binary

def save_binary(variable, pathfile):
    import pickle
    import tempfile
    import shutil
    import os
    # Write to a temporary file first
    dir_name = os.path.dirname(pathfile)
    os.makedirs(dir_name, exist_ok=True)
    try:
        with tempfile.NamedTemporaryFile(dir=dir_name, delete=False) as tmp_file:
            pickle.dump(variable, tmp_file)
            temp_name = tmp_file.name
        # If pickle.dump succeeds, replace the original file
        shutil.move(temp_name, pathfile)
    except Exception as exc:
        print("Error Saving Binary for pathfile:", pathfile)
        print(exc)
        # Clean up temp file if exists
        if 'temp_name' in locals() and os.path.exists(temp_name):
            os.remove(temp_name)